Script for lock in:

 parameter: r (resistance)

 - Set V (Read) (can calculate I=V/r)
 - Set frequency (Read)
 - Read R ($\sqrt(X^2+Y^2)$), $\theta$ (r = R / I)
 - Voltage: A_B (Read)
 - Other: time constant, input range, sensitivity

 Read lock in (R, $\theta$) ever x seconds 
 Save to text file (also need to read temperatures from MXC) as MXC Temperature, Resistance (Ohms), Theta

Read MXC:
LakeShore

In [2]:
import pyvisa
from pyvisa import VisaIOError
from datetime import datetime
import Lockin_class as LC
import pandas as pd
%matplotlib tk
import matplotlib.pyplot as plt

In [2]:
rm = pyvisa.ResourceManager()

for s in rm.list_resources():
    try:
        lockin = rm.open_resource(s)
        print(s, lockin.query('*IDN?').strip())
    except VisaIOError:
        continue

USB0::0xB506::0x2000::004505::INSTR Stanford_Research_Systems,SR865A,004505,1.55


In [2]:
rm = pyvisa.ResourceManager()
lockin = rm.open_resource('USB0::0xB506::0x2000::004505::INSTR')  # where the lockin is

if 0:  # to use LakeShore with GPIB
    temp_reader = rm.open_resource('GPIB1::12::INSTR')  # where the LakeShore is
elif 0:  # to use LakeShore with Ethernet
    ip_address = "192.168.1.3"
    port = 7777

    visa_resource = f"TCPIP::{ip_address}::{port}::SOCKET"

    rm = pyvisa.ResourceManager()
    temp_reader = rm.open_resource(visa_resource)  # where the LakeShore is

    temp_reader.read_termination = '\n'
    temp_reader.write_termination = '\n'
    temp_reader.timeout = 5000  # milliseconds
else:  # to use SRS CTC100 with USB
    temp_reader = rm.open_resource('ASRL10::INSTR')  # where the SRS CTC100 is
    temp_reader.baud_rate = 19200

print("Connected to:", lockin.query("*IDN?").strip())
print("Connected to:", temp_reader.query("*IDN?").strip())

Connected to: Stanford_Research_Systems,SR865A,004505,1.55
Connected to: Stanford Research Systems, CTC100 Cryogenic Temperature Controller, 153490, version 4.41


In [3]:
Calibration = LC.lockin(lockin, temp_reader, Rbias=10e6)
Calibration.initialize(
    v = 1,  # input V
    f = 1.17e3,  # input frequency
    n = 1  # signal sensitivity (0: A, 1: A - B)
)

In [ ]:
Calibration.log_data(
    "C:\\Users\\gerar\\OneDrive\\Documents\\RTD_2.txt",
    is_lakeshore=False,
    sampling_spacing=1,
    init_sleep=0,
    hours=3
    # seconds=5
)

In [5]:
data = pd.read_csv("C:\\Users\\gerar\\OneDrive\\Documents\\RTD.txt", delimiter = '\t')
data_1 = pd.read_csv("C:\\Users\\gerar\\OneDrive\\Documents\\RTD_2.txt", delimiter = '\t')
data_2 = pd.read_csv("C:\\Users\\gerar\\OneDrive\\Documents\\RTD_Warm_Up.txt", delimiter = '\t')
data_3 = pd.read_csv("C:\\Users\\gerar\\OneDrive\\Documents\\RTD_Warm_Up_2.txt", delimiter = '\t')
data_4 = pd.read_csv("C:\\Users\\gerar\\OneDrive\\Documents\\RTD_Warm_Up_3.txt", delimiter = '\t')

In [9]:
plt.scatter(data['Temperature (K)'], data['Resistance (Ohms)'], s=1)
plt.scatter(data_1['Temperature (K)'], data_1['Resistance (Ohms)'], s=1)
plt.scatter(data_4['Temperature (K)'], data_4['Resistance (Ohms)'], s=1)
plt.grid()
plt.xlabel("Temperature (K)")
plt.ylabel("Resistance (Ohms)")

Text(0, 0.5, 'Resistance (Ohms)')